# rabi posterior learning
See the repository README for the paper configurations and cached-data reproduction commands.


In [ ]:
from __future__ import annotations

# Matplotlib backend is selected by the caller.

import hashlib
import importlib
import json
import math
import os
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import LogNorm
from matplotlib.ticker import LogLocator, NullFormatter


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "dm_search").exists():
            return candidate
    raise RuntimeError("Could not find repository root")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import dm_search.rabi as _rabi_module  # noqa: E402
_rabi_module = importlib.reload(_rabi_module)

from dm_search.rabi import (  # noqa: E402
    RabiExperimentConfig,
    RabiParticleSpec,
    RabiReadoutConfig,
    build_rabi_particle_grid,
    rabi_probability_one,
)
try:  # noqa: E402
    import dm_search.numba_rabi as _numba_rabi_module
    _numba_rabi_module = importlib.reload(_numba_rabi_module)
    from dm_search.numba_rabi import run_rabi_sphere_learning_trajectory_numba
except Exception:  # pragma: no cover - notebook fallback path
    run_rabi_sphere_learning_trajectory_numba = None


plt.style.use("seaborn-v0_8-whitegrid")

# Shared manuscript typography and frameless legends.
plt.style.use(REPO_ROOT / "plot_final" / "rabi_paper.mplstyle")

# Publication baseline: do not use smaller type at this figure-to-page scale.
plt.rcParams.update({
    "font.size": 14,
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 13,
    "figure.titlesize": 14,
    "axes.grid": False,
})


In [ ]:

# Parameters.  Override with environment variables for offline runs.
RUN_PRESET = os.environ.get("RABI_LEARNING_PRESET", "final")
DETECTOR_PROFILE = os.environ.get("RABI_LEARNING_DETECTOR_PROFILE", "current")
DETECTOR_PROFILES = {
    # Main-text candidate: realistic readout and the period-scan winner L=1 (pure IG).
    "current": dict(contrast=0.99, readout_error=0.005, readout_objective="information_gain", periodic_hybrid_period=1, phi_r_true=0.19, seed_signal=7000),
    # Appendix benchmark: the earlier high-fidelity L=8 IG/Helstrom trajectory.
    "high_fidelity": dict(contrast=0.999, readout_error=0.0005, readout_objective="periodic_hybrid", periodic_hybrid_period=8, phi_r_true=0.19, seed_signal=7020),
}
if DETECTOR_PROFILE not in DETECTOR_PROFILES:
    raise ValueError(f"unknown RABI_LEARNING_DETECTOR_PROFILE={DETECTOR_PROFILE!r}")
D = DETECTOR_PROFILES[DETECTOR_PROFILE]
READOUT_SPACE = "sphere"
READOUT_OPTIMIZATION = "continuous"
READOUT_OBJECTIVE = D["readout_objective"]
PRESETS = {
    "quick": dict(n_shots=128, amplitude_count=18, phase_count=36, readout_space=READOUT_SPACE, readout_optimization=READOUT_OPTIMIZATION, readout_objective=READOUT_OBJECTIVE, candidate_angle_count=16, candidate_polar_count=5, mixed_phase_concentration=0.9, guarded_information_fraction=0.4, guarded_information_floor=0.0, hybrid_information_weight_min=0.0, hybrid_information_weight_max=1.0, hybrid_r_start=0.2, hybrid_r_stop=0.9, periodic_hybrid_period=D["periodic_hybrid_period"], readout_switch_shot=128, blend_lambda_max=1.0, blend_r_start=0.2, blend_r_stop=0.9, snapshots="0,2,8,32,128"),
    "draft": dict(n_shots=512, amplitude_count=24, phase_count=48, readout_space=READOUT_SPACE, readout_optimization=READOUT_OPTIMIZATION, readout_objective=READOUT_OBJECTIVE, candidate_angle_count=16, candidate_polar_count=5, mixed_phase_concentration=0.9, guarded_information_fraction=0.4, guarded_information_floor=0.0, hybrid_information_weight_min=0.0, hybrid_information_weight_max=1.0, hybrid_r_start=0.2, hybrid_r_stop=0.9, periodic_hybrid_period=D["periodic_hybrid_period"], readout_switch_shot=512, blend_lambda_max=1.0, blend_r_start=0.2, blend_r_stop=0.9, snapshots="0,4,16,64,256,1024"),
    "final": dict(n_shots=4096, amplitude_count=48, phase_count=96, readout_space=READOUT_SPACE, readout_optimization=READOUT_OPTIMIZATION, readout_objective=READOUT_OBJECTIVE, candidate_angle_count=16, candidate_polar_count=7, mixed_phase_concentration=0.9, guarded_information_fraction=0.4, guarded_information_floor=0.0, hybrid_information_weight_min=0.0, hybrid_information_weight_max=1.0, hybrid_r_start=0.2, hybrid_r_stop=0.9, periodic_hybrid_period=D["periodic_hybrid_period"], readout_switch_shot=4096, blend_lambda_max=1.0, blend_r_start=0.2, blend_r_stop=0.9, snapshots="0,16,64,128,256,512,1024,2048,4096"),
}
if RUN_PRESET not in PRESETS:
    raise ValueError(f"unknown RABI_LEARNING_PRESET={RUN_PRESET!r}")
P = PRESETS[RUN_PRESET]

N_SHOTS = int(os.environ.get("RABI_LEARNING_N_SHOTS", P["n_shots"]))
PHI_R_TRUE = float(os.environ.get("RABI_LEARNING_PHI_R", D["phi_r_true"]))
TRUE_DRIVE_PHASE = float(os.environ.get("RABI_LEARNING_TRUE_DRIVE_PHASE", "0.7"))
TRUE_DETUNING = 0.0
SENSING_TIME = float(os.environ.get("RABI_LEARNING_SENSING_TIME", "1.0"))
DEAD_TIME = float(os.environ.get("RABI_LEARNING_DEAD_TIME", "0.0"))
T1_TIME = float(os.environ.get("RABI_LEARNING_T1", str(SENSING_TIME)))
T2_TIME = float(os.environ.get("RABI_LEARNING_T2", str(SENSING_TIME)))
CONTRAST = float(os.environ.get("RABI_LEARNING_CONTRAST", D["contrast"]))
READOUT_ERROR = float(os.environ.get("RABI_LEARNING_READOUT_ERROR", D["readout_error"]))
INITIAL_H1_PROBABILITY = float(os.environ.get("RABI_LEARNING_INITIAL_H1", "0.5"))

PHASE_AMPLITUDE_MIN = float(os.environ.get("RABI_LEARNING_AMPLITUDE_MIN", "0.02"))
PHASE_AMPLITUDE_MAX = float(os.environ.get("RABI_LEARNING_AMPLITUDE_MAX", "1.2"))
PHASE_AMPLITUDE_COUNT = int(os.environ.get("RABI_LEARNING_AMPLITUDE_COUNT", P["amplitude_count"]))
PHASE_AMPLITUDE_PRIOR = os.environ.get("RABI_LEARNING_AMPLITUDE_PRIOR", "log_uniform")
DRIVE_PHASE_COUNT = int(os.environ.get("RABI_LEARNING_PHASE_COUNT", P["phase_count"]))
READOUT_SPACE = os.environ.get("RABI_LEARNING_READOUT_SPACE", P.get("readout_space", "transverse"))
if READOUT_SPACE not in {"transverse", "sphere"}:
    raise ValueError("RABI_LEARNING_READOUT_SPACE must be transverse or sphere")
READOUT_OPTIMIZATION = os.environ.get("RABI_LEARNING_READOUT_OPTIMIZATION", P["readout_optimization"])
if READOUT_OPTIMIZATION not in {"grid", "continuous"}:
    raise ValueError("RABI_LEARNING_READOUT_OPTIMIZATION must be grid or continuous")
READOUT_OBJECTIVE = os.environ.get("RABI_LEARNING_READOUT_OBJECTIVE", P["readout_objective"])
if READOUT_OBJECTIVE not in {"information_gain", "helstrom", "guarded_helstrom", "mixed", "mixed_guarded", "periodic_hybrid", "periodic_bayes_hybrid", "periodic_stein_hybrid", "bayes_drift", "stein_drift", "hybrid", "blended"}:
    raise ValueError("RABI_LEARNING_READOUT_OBJECTIVE must be information_gain, helstrom, guarded_helstrom, mixed, mixed_guarded, periodic_hybrid, periodic_bayes_hybrid, periodic_stein_hybrid, bayes_drift, stein_drift, hybrid, or blended")
MIXED_PHASE_CONCENTRATION = float(os.environ.get("RABI_LEARNING_MIXED_PHASE_CONCENTRATION", P.get("mixed_phase_concentration", 0.9)))
if not (0.0 <= MIXED_PHASE_CONCENTRATION <= 1.0):
    raise ValueError("RABI_LEARNING_MIXED_PHASE_CONCENTRATION must be between 0 and 1")
GUARDED_INFORMATION_FRACTION = float(os.environ.get("RABI_LEARNING_GUARDED_INFORMATION_FRACTION", P.get("guarded_information_fraction", 0.25)))
if not (0.0 <= GUARDED_INFORMATION_FRACTION <= 1.0):
    raise ValueError("RABI_LEARNING_GUARDED_INFORMATION_FRACTION must be between 0 and 1")
GUARDED_INFORMATION_FLOOR = float(os.environ.get("RABI_LEARNING_GUARDED_INFORMATION_FLOOR", P.get("guarded_information_floor", 0.0)))
if GUARDED_INFORMATION_FLOOR < 0.0:
    raise ValueError("RABI_LEARNING_GUARDED_INFORMATION_FLOOR must be non-negative")
HYBRID_INFORMATION_WEIGHT_MIN = float(os.environ.get("RABI_LEARNING_HYBRID_INFORMATION_WEIGHT_MIN", P.get("hybrid_information_weight_min", 0.0)))
HYBRID_INFORMATION_WEIGHT_MAX = float(os.environ.get("RABI_LEARNING_HYBRID_INFORMATION_WEIGHT_MAX", P.get("hybrid_information_weight_max", 1.0)))
HYBRID_R_START = float(os.environ.get("RABI_LEARNING_HYBRID_R_START", P.get("hybrid_r_start", 0.0)))
HYBRID_R_STOP = float(os.environ.get("RABI_LEARNING_HYBRID_R_STOP", P.get("hybrid_r_stop", 1.0)))
if not (0.0 <= HYBRID_INFORMATION_WEIGHT_MIN <= HYBRID_INFORMATION_WEIGHT_MAX <= 1.0 and 0.0 <= HYBRID_R_START < HYBRID_R_STOP <= 1.0):
    raise ValueError("hybrid parameters must satisfy 0 <= weight_min <= weight_max <= 1 and 0 <= R_start < R_stop <= 1")
PERIODIC_HYBRID_PERIOD = int(os.environ.get("RABI_LEARNING_PERIODIC_HYBRID_PERIOD", P.get("periodic_hybrid_period", 4)))
if PERIODIC_HYBRID_PERIOD <= 0:
    raise ValueError("RABI_LEARNING_PERIODIC_HYBRID_PERIOD must be positive")
READOUT_SWITCH_SHOT = int(os.environ.get("RABI_LEARNING_READOUT_SWITCH_SHOT", P.get("readout_switch_shot", 0)))
if READOUT_SWITCH_SHOT < 0:
    raise ValueError("RABI_LEARNING_READOUT_SWITCH_SHOT must be non-negative")
BLEND_LAMBDA_MAX = float(os.environ.get("RABI_LEARNING_BLEND_LAMBDA_MAX", P.get("blend_lambda_max", 1.0)))
BLEND_R_START = float(os.environ.get("RABI_LEARNING_BLEND_R_START", P.get("blend_r_start", 0.2)))
BLEND_R_STOP = float(os.environ.get("RABI_LEARNING_BLEND_R_STOP", P.get("blend_r_stop", 0.9)))
if BLEND_LAMBDA_MAX < 0.0 or not (0.0 <= BLEND_R_START < BLEND_R_STOP <= 1.0):
    raise ValueError("blend parameters must satisfy lambda_max >= 0 and 0 <= R_start < R_stop <= 1")
candidate_count_text = os.environ.get("RABI_LEARNING_CANDIDATE_COUNT")
CANDIDATE_ANGLE_COUNT = int(candidate_count_text) if candidate_count_text is not None else int(P.get("candidate_angle_count", 16))
if CANDIDATE_ANGLE_COUNT <= 0:
    raise ValueError("RABI_LEARNING_CANDIDATE_COUNT must be positive")
READOUT_AZIMUTH_PERIOD = math.pi
CANDIDATE_ANGLES = tuple(np.linspace(0.0, READOUT_AZIMUTH_PERIOD, CANDIDATE_ANGLE_COUNT, endpoint=False))
CANDIDATE_POLAR_COUNT = int(os.environ.get("RABI_LEARNING_CANDIDATE_POLAR_COUNT", P.get("candidate_polar_count", 1)))
if CANDIDATE_POLAR_COUNT <= 0:
    raise ValueError("RABI_LEARNING_CANDIDATE_POLAR_COUNT must be positive")
if READOUT_SPACE == "transverse":
    CANDIDATE_POLARS = (0.5 * math.pi,)
else:
    CANDIDATE_POLARS = tuple(np.linspace(0.0, math.pi, CANDIDATE_POLAR_COUNT))

snapshot_text = os.environ.get("RABI_LEARNING_SNAPSHOT_SHOTS", P["snapshots"])
SNAPSHOT_SHOTS = tuple(int(x) for x in snapshot_text.split(",") if x.strip())
if 0 not in SNAPSHOT_SHOTS:
    SNAPSHOT_SHOTS = (0,) + SNAPSHOT_SHOTS
if N_SHOTS not in SNAPSHOT_SHOTS:
    SNAPSHOT_SHOTS = SNAPSHOT_SHOTS + (N_SHOTS,)
SNAPSHOT_SHOTS = tuple(sorted(set(s for s in SNAPSHOT_SHOTS if 0 <= s <= N_SHOTS)))

SEED_SIGNAL = int(os.environ.get("RABI_LEARNING_SEED_SIGNAL", D["seed_signal"]))
SEED_BACKGROUND = int(os.environ.get("RABI_LEARNING_SEED_BACKGROUND", "7326"))
USE_CACHE = os.environ.get("RABI_LEARNING_USE_CACHE", "1") != "0"
FORCE_RECOMPUTE = os.environ.get("RABI_LEARNING_FORCE_RECOMPUTE", "0") == "1"
# FORCE_RECOMPUTE = True
FAST_DETECTION_OBJECTIVES = os.environ.get("RABI_LEARNING_FAST_DETECTION", "1") != "0"
TRAJECTORY_BACKEND = os.environ.get("RABI_LEARNING_TRAJECTORY_BACKEND", "numba")
if TRAJECTORY_BACKEND not in {"python", "numba"}:
    raise ValueError("RABI_LEARNING_TRAJECTORY_BACKEND must be python or numba")

SOURCE_TRAJECTORY_STRATEGY = None
if READOUT_SPACE == "sphere" and READOUT_OPTIMIZATION == "continuous":
    if READOUT_OBJECTIVE == "information_gain":
        SOURCE_TRAJECTORY_STRATEGY = "adaptive_sphere_continuous"
    elif READOUT_OBJECTIVE == "bayes_drift":
        SOURCE_TRAJECTORY_STRATEGY = "adaptive_bayes_sphere_continuous"
    elif READOUT_OBJECTIVE == "periodic_hybrid":
        SOURCE_TRAJECTORY_STRATEGY = f"periodic_hybrid_sphere_continuous_{PERIODIC_HYBRID_PERIOD}"
    elif READOUT_OBJECTIVE == "periodic_bayes_hybrid":
        SOURCE_TRAJECTORY_STRATEGY = f"periodic_bayes_hybrid_sphere_continuous_{PERIODIC_HYBRID_PERIOD}"
    elif READOUT_OBJECTIVE == "periodic_stein_hybrid":
        SOURCE_TRAJECTORY_STRATEGY = f"periodic_stein_hybrid_sphere_continuous_{PERIODIC_HYBRID_PERIOD}"
USE_SOURCE_TRAJECTORY = TRAJECTORY_BACKEND == "numba" and SOURCE_TRAJECTORY_STRATEGY is not None and run_rabi_sphere_learning_trajectory_numba is not None
CACHE_VERSION = "posterior-learning-v16-detector-profiles"
OUTPUT_BASENAME = "rabi_posterior_learning"
OUTPUT_TAG = f"{RUN_PRESET}_{DETECTOR_PROFILE}"
CACHE_DIR = REPO_ROOT / "plot_final" / "cache"
FIGURE_DIR = REPO_ROOT / "plot_final" / "figure"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
LEGACY_HIGH_FIDELITY_SETUP_PATH = FIGURE_DIR / f"{OUTPUT_BASENAME}_final_setup.json"

print(f"preset={RUN_PRESET}, detector profile={DETECTOR_PROFILE}, n_shots={N_SHOTS}, Phi_R={PHI_R_TRUE}, true phase={TRUE_DRIVE_PHASE}")
print(f"detector: C={CONTRAST:g}, epsilon={READOUT_ERROR:g}")
print(f"posterior grid: A={PHASE_AMPLITUDE_COUNT}, Delta=known 0, phase={DRIVE_PHASE_COUNT}")
switch_note = ""
mixed_note = f", switch R={MIXED_PHASE_CONCENTRATION}" if READOUT_OBJECTIVE in {"mixed", "mixed_guarded"} else ""
guard_note = f", IG guard={GUARDED_INFORMATION_FRACTION:g} max + {GUARDED_INFORMATION_FLOOR:g}" if READOUT_OBJECTIVE in {"guarded_helstrom", "mixed_guarded"} else ""
hybrid_note = f", hybrid wIG={HYBRID_INFORMATION_WEIGHT_MIN:g}-{HYBRID_INFORMATION_WEIGHT_MAX:g}, R={HYBRID_R_START}-{HYBRID_R_STOP}" if READOUT_OBJECTIVE == "hybrid" else ""
periodic_note = f", IG every {PERIODIC_HYBRID_PERIOD} shots" if READOUT_OBJECTIVE in {"periodic_hybrid", "periodic_bayes_hybrid", "periodic_stein_hybrid"} else ""
blend_note = f", blend lambda={BLEND_LAMBDA_MAX}, R={BLEND_R_START}-{BLEND_R_STOP}" if READOUT_OBJECTIVE == "blended" else ""
source_note = f", source strategy={SOURCE_TRAJECTORY_STRATEGY}" if USE_SOURCE_TRAJECTORY else ""
print(f"readout space={READOUT_SPACE}, optimization={READOUT_OPTIMIZATION}, objective={READOUT_OBJECTIVE}{switch_note}{mixed_note}{guard_note}{hybrid_note}{periodic_note}{blend_note}, backend={TRAJECTORY_BACKEND}, use source={USE_SOURCE_TRAJECTORY}{source_note}, fast detection={FAST_DETECTION_OBJECTIVES}, coarse polar={len(CANDIDATE_POLARS)}, coarse azimuth={CANDIDATE_ANGLE_COUNT}, snapshots={SNAPSHOT_SHOTS}")


In [ ]:

particles = RabiParticleSpec(
    particle_mode="grid",
    phase_amplitude_min=PHASE_AMPLITUDE_MIN,
    phase_amplitude_max=PHASE_AMPLITUDE_MAX,
    phase_amplitude_count=PHASE_AMPLITUDE_COUNT,
    phase_amplitude_prior=PHASE_AMPLITUDE_PRIOR,
    detuning_min=0.0,
    detuning_max=0.0,
    detuning_count=0,
    drive_phase_count=DRIVE_PHASE_COUNT,
)
config = RabiExperimentConfig(
    n_shots=N_SHOTS,
    sensing_time=SENSING_TIME,
    dead_time=DEAD_TIME,
    true_phase_amplitude=PHI_R_TRUE,
    true_detuning=TRUE_DETUNING,
    true_drive_phase=TRUE_DRIVE_PHASE,
    initial_h1_probability=INITIAL_H1_PROBABILITY,
    readout=RabiReadoutConfig(contrast=CONTRAST, readout_error=READOUT_ERROR, t1=T1_TIME, t2=T2_TIME),
    particles=particles,
    candidate_angles=CANDIDATE_ANGLES,
    candidate_polars=CANDIDATE_POLARS,
    seed=SEED_SIGNAL,
)
grid = build_rabi_particle_grid(config.particles)
print("particle shape:", grid.shape, "particle count:", grid.prior_weights.size)

setup = {
    "cache_version": CACHE_VERSION,
    "run_preset": RUN_PRESET,
    "detector_profile": DETECTOR_PROFILE,
    "n_shots": N_SHOTS,
    "phi_r_true": PHI_R_TRUE,
    "true_drive_phase": TRUE_DRIVE_PHASE,
    "true_detuning": TRUE_DETUNING,
    "sensing_time": SENSING_TIME,
    "dead_time": DEAD_TIME,
    "t1_time": T1_TIME,
    "t2_time": T2_TIME,
    "contrast": CONTRAST,
    "readout_error": READOUT_ERROR,
    "initial_h1_probability": INITIAL_H1_PROBABILITY,
    "phase_amplitude_min": PHASE_AMPLITUDE_MIN,
    "phase_amplitude_max": PHASE_AMPLITUDE_MAX,
    "phase_amplitude_count": PHASE_AMPLITUDE_COUNT,
    "phase_amplitude_prior": PHASE_AMPLITUDE_PRIOR,
    "drive_phase_count": DRIVE_PHASE_COUNT,
    "readout_space": READOUT_SPACE,
    "readout_azimuth_period": READOUT_AZIMUTH_PERIOD,
    "readout_optimization": READOUT_OPTIMIZATION,
    "readout_objective": READOUT_OBJECTIVE,
    "mixed_phase_concentration": MIXED_PHASE_CONCENTRATION,
    "guarded_information_fraction": GUARDED_INFORMATION_FRACTION,
    "guarded_information_floor": GUARDED_INFORMATION_FLOOR,
    "hybrid_information_weight_min": HYBRID_INFORMATION_WEIGHT_MIN,
    "hybrid_information_weight_max": HYBRID_INFORMATION_WEIGHT_MAX,
    "hybrid_r_start": HYBRID_R_START,
    "hybrid_r_stop": HYBRID_R_STOP,
    "periodic_hybrid_period": PERIODIC_HYBRID_PERIOD,
    "readout_switch_shot": READOUT_SWITCH_SHOT,
    "blend_lambda_max": BLEND_LAMBDA_MAX,
    "blend_r_start": BLEND_R_START,
    "blend_r_stop": BLEND_R_STOP,
    "candidate_angle_count": CANDIDATE_ANGLE_COUNT,
    "candidate_polar_count": CANDIDATE_POLAR_COUNT,
    "trajectory_backend": TRAJECTORY_BACKEND,
    "use_source_trajectory": USE_SOURCE_TRAJECTORY,
    "source_trajectory_strategy": SOURCE_TRAJECTORY_STRATEGY,
    "snapshot_shots": SNAPSHOT_SHOTS,
    "seed_signal": SEED_SIGNAL,
    "seed_background": SEED_BACKGROUND,
    "fast_detection_objectives": FAST_DETECTION_OBJECTIVES,
    "legacy_cache_fallback_enabled": DETECTOR_PROFILE == "high_fidelity",
}
setup_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{OUTPUT_TAG}_setup.json"
setup_path.write_text(json.dumps(setup, indent=2, sort_keys=True), encoding="utf-8")
print(f"saved {setup_path}")


In [ ]:

EPS = 1.0e-12


def binary_entropy(probability):
    p = np.clip(probability, EPS, 1.0 - EPS)
    return -(p * np.log(p) + (1.0 - p) * np.log(1.0 - p))


def binary_kl(probability_one, reference_one):
    p = np.clip(probability_one, EPS, 1.0 - EPS)
    q = np.clip(reference_one, EPS, 1.0 - EPS)
    return p * np.log(p / q) + (1.0 - p) * np.log((1.0 - p) / (1.0 - q))


def h0_probability_one(axis, readout_angle, cfg):
    return float(
        np.clip(
            rabi_probability_one(0.0, 0.0, 0.0, 0.0, cfg.sensing_time, axis, readout_angle, cfg.readout),
            EPS,
            1.0 - EPS,
        )
    )


def probability_to_bloch_projection(probability, readout):
    ideal = (np.asarray(probability, dtype=float) - readout.readout_error) / (1.0 - 2.0 * readout.readout_error)
    return (2.0 * ideal - 1.0) / readout.contrast


def bloch_projection_to_probability(projection, readout):
    ideal = 0.5 * (1.0 + readout.contrast * np.asarray(projection, dtype=float))
    probability = readout.readout_error + (1.0 - 2.0 * readout.readout_error) * ideal
    return np.clip(probability, EPS, 1.0 - EPS)


def canonical_readout_action(readout_polar, readout_angle):
    if READOUT_SPACE == "transverse":
        return 0.5 * math.pi, float(readout_angle) % math.pi
    x = math.sin(float(readout_polar)) * math.cos(float(readout_angle))
    y = math.sin(float(readout_polar)) * math.sin(float(readout_angle))
    z = math.cos(float(readout_polar))
    if z < 0.0:
        x, y, z = -x, -y, -z
    norm = math.sqrt(x * x + y * y + z * z)
    if norm <= 0.0:
        return 0.0, 0.0
    x, y, z = x / norm, y / norm, z / norm
    polar = math.acos(float(np.clip(z, -1.0, 1.0)))
    azimuth = math.atan2(y, x) % (2.0 * math.pi)
    if azimuth >= math.pi:
        polar = math.pi - polar
        azimuth -= math.pi
    return polar, azimuth


def particle_probabilities_for_action(cfg, particle_grid, readout_polar, readout_angle, start_time):
    readout_polar, readout_angle = canonical_readout_action(readout_polar, readout_angle)
    z_probability = rabi_probability_one(
        particle_grid.phase_amplitudes,
        particle_grid.detunings,
        particle_grid.drive_phases,
        start_time,
        cfg.sensing_time,
        "z",
        0.0,
        cfg.readout,
    )
    transverse_probability = rabi_probability_one(
        particle_grid.phase_amplitudes,
        particle_grid.detunings,
        particle_grid.drive_phases,
        start_time,
        cfg.sensing_time,
        "transverse",
        readout_angle,
        cfg.readout,
    )
    z_projection = probability_to_bloch_projection(z_probability, cfg.readout)
    transverse_projection = probability_to_bloch_projection(transverse_probability, cfg.readout)
    projection = math.cos(readout_polar) * z_projection + math.sin(readout_polar) * transverse_projection
    return np.asarray(bloch_projection_to_probability(projection, cfg.readout), dtype=float)


def h0_probability_one_for_action(readout_polar, readout_angle, cfg):
    readout_polar, readout_angle = canonical_readout_action(readout_polar, readout_angle)
    z_probability = h0_probability_one("z", 0.0, cfg)
    transverse_probability = h0_probability_one("transverse", readout_angle, cfg)
    z_projection = probability_to_bloch_projection(z_probability, cfg.readout)
    transverse_projection = probability_to_bloch_projection(transverse_probability, cfg.readout)
    projection = math.cos(float(readout_polar)) * z_projection + math.sin(float(readout_polar)) * transverse_projection
    return float(bloch_projection_to_probability(projection, cfg.readout))


def true_probability_one_for_action(cfg, readout_polar, readout_angle, start_time):
    readout_polar, readout_angle = canonical_readout_action(readout_polar, readout_angle)
    z_probability = rabi_probability_one(
        cfg.true_phase_amplitude,
        cfg.true_detuning,
        cfg.true_drive_phase,
        start_time,
        cfg.sensing_time,
        "z",
        0.0,
        cfg.readout,
    )
    transverse_probability = rabi_probability_one(
        cfg.true_phase_amplitude,
        cfg.true_detuning,
        cfg.true_drive_phase,
        start_time,
        cfg.sensing_time,
        "transverse",
        readout_angle,
        cfg.readout,
    )
    z_projection = probability_to_bloch_projection(z_probability, cfg.readout)
    transverse_projection = probability_to_bloch_projection(transverse_probability, cfg.readout)
    projection = math.cos(float(readout_polar)) * z_projection + math.sin(float(readout_polar)) * transverse_projection
    return float(bloch_projection_to_probability(projection, cfg.readout))


def transverse_particle_probabilities(cfg, particle_grid, readout_angle, start_time):
    return np.asarray(
        np.clip(
            rabi_probability_one(
                particle_grid.phase_amplitudes,
                particle_grid.detunings,
                particle_grid.drive_phases,
                start_time,
                cfg.sensing_time,
                "transverse",
                readout_angle,
                cfg.readout,
            ),
            EPS,
            1.0 - EPS,
        ),
        dtype=float,
    )


def angle_utility(cfg, particle_grid, weights, h1_probability, readout_polar, readout_angle, start_time):
    h0_one = h0_probability_one_for_action(readout_polar, readout_angle, cfg)
    h0_entropy = binary_entropy(h0_one)
    particle_one = particle_probabilities_for_action(cfg, particle_grid, readout_polar, readout_angle, start_time)
    mixture_one = float(np.dot(weights, particle_one))
    h1_conditional_entropy = float(np.dot(weights, binary_entropy(particle_one)))
    predictive_one = (1.0 - h1_probability) * h0_one + h1_probability * mixture_one
    conditional_entropy = (1.0 - h1_probability) * h0_entropy + h1_probability * h1_conditional_entropy
    utility = float(binary_entropy(predictive_one) - conditional_entropy)
    return utility, particle_one, mixture_one


def angle_bayes_drift(cfg, particle_grid, weights, readout_polar, readout_angle, start_time):
    h0_one = h0_probability_one_for_action(readout_polar, readout_angle, cfg)
    particle_one = particle_probabilities_for_action(cfg, particle_grid, readout_polar, readout_angle, start_time)
    mixture_one = float(np.dot(weights, particle_one))
    drift = float(binary_kl(mixture_one, h0_one))
    return drift, particle_one, mixture_one


def angle_stein_drift(cfg, particle_grid, weights, readout_polar, readout_angle, start_time):
    h0_one = h0_probability_one_for_action(readout_polar, readout_angle, cfg)
    particle_one = particle_probabilities_for_action(cfg, particle_grid, readout_polar, readout_angle, start_time)
    mixture_one = float(np.dot(weights, particle_one))
    drift = float(binary_kl(h0_one, mixture_one))
    return drift, particle_one, mixture_one


def angle_helstrom(cfg, particle_grid, weights, readout_polar, readout_angle, start_time):
    h0_one = h0_probability_one_for_action(readout_polar, readout_angle, cfg)
    particle_one = particle_probabilities_for_action(cfg, particle_grid, readout_polar, readout_angle, start_time)
    mixture_one = float(np.dot(weights, particle_one))
    separation = abs(mixture_one - h0_one)
    return separation, particle_one, mixture_one


def posterior_bloch_moments_for_detection(cfg, particle_grid, weights, start_time):
    z_probability = rabi_probability_one(
        particle_grid.phase_amplitudes,
        particle_grid.detunings,
        particle_grid.drive_phases,
        start_time,
        cfg.sensing_time,
        "z",
        0.0,
        cfg.readout,
    )
    x_probability = rabi_probability_one(
        particle_grid.phase_amplitudes,
        particle_grid.detunings,
        particle_grid.drive_phases,
        start_time,
        cfg.sensing_time,
        "transverse",
        0.0,
        cfg.readout,
    )
    y_probability = rabi_probability_one(
        particle_grid.phase_amplitudes,
        particle_grid.detunings,
        particle_grid.drive_phases,
        start_time,
        cfg.sensing_time,
        "transverse",
        0.5 * math.pi,
        cfg.readout,
    )
    z_projection = probability_to_bloch_projection(z_probability, cfg.readout)
    x_projection = probability_to_bloch_projection(x_probability, cfg.readout)
    y_projection = probability_to_bloch_projection(y_probability, cfg.readout)
    return np.array(
        [
            float(np.dot(weights, x_projection)),
            float(np.dot(weights, y_projection)),
            float(np.dot(weights, z_projection)),
        ],
        dtype=float,
    )


def detection_probabilities_from_moments(cfg, moments, readout_polar, readout_angle):
    readout_polar, readout_angle = canonical_readout_action(readout_polar, readout_angle)
    sin_polar = math.sin(float(readout_polar))
    cos_polar = math.cos(float(readout_polar))
    projection_h1 = (
        sin_polar * math.cos(float(readout_angle)) * float(moments[0])
        + sin_polar * math.sin(float(readout_angle)) * float(moments[1])
        + cos_polar * float(moments[2])
    )
    projection_h0 = -cos_polar
    mixture_one = float(bloch_projection_to_probability(projection_h1, cfg.readout))
    h0_one = float(bloch_projection_to_probability(projection_h0, cfg.readout))
    return h0_one, mixture_one


def detection_objective_from_moments(cfg, moments, readout_polar, readout_angle, objective):
    h0_one, mixture_one = detection_probabilities_from_moments(cfg, moments, readout_polar, readout_angle)
    if objective == "helstrom":
        return abs(mixture_one - h0_one), mixture_one
    if objective == "bayes_drift":
        return float(binary_kl(mixture_one, h0_one)), mixture_one
    if objective == "stein_drift":
        return float(binary_kl(h0_one, mixture_one)), mixture_one
    raise ValueError("fast detection objective must be helstrom, bayes_drift, or stein_drift")


def angle_guarded_helstrom(cfg, particle_grid, weights, h1_probability, readout_polar, readout_angle, start_time, information_threshold):
    h0_one = h0_probability_one_for_action(readout_polar, readout_angle, cfg)
    h0_entropy = binary_entropy(h0_one)
    particle_one = particle_probabilities_for_action(cfg, particle_grid, readout_polar, readout_angle, start_time)
    mixture_one = float(np.dot(weights, particle_one))
    h1_conditional_entropy = float(np.dot(weights, binary_entropy(particle_one)))
    predictive_one = (1.0 - h1_probability) * h0_one + h1_probability * mixture_one
    conditional_entropy = (1.0 - h1_probability) * h0_entropy + h1_probability * h1_conditional_entropy
    information_gain = float(binary_entropy(predictive_one) - conditional_entropy)
    separation = float(abs(mixture_one - h0_one))
    valid = information_gain + 1.0e-14 >= information_threshold
    score = separation if valid else -np.inf
    return score, separation, information_gain, particle_one, mixture_one


def phase_concentration(particle_grid, weights):
    z = np.dot(weights, np.exp(1j * particle_grid.drive_phases))
    return float(np.abs(z))


def blend_lambda_from_concentration(concentration):
    if concentration <= BLEND_R_START:
        return BLEND_LAMBDA_MAX
    if concentration >= BLEND_R_STOP:
        return 0.0
    return float(BLEND_LAMBDA_MAX * (BLEND_R_STOP - concentration) / (BLEND_R_STOP - BLEND_R_START))


def readout_blend_lambda(particle_grid, weights):
    return blend_lambda_from_concentration(phase_concentration(particle_grid, weights))


def hybrid_information_weight_from_concentration(concentration):
    if concentration <= HYBRID_R_START:
        return HYBRID_INFORMATION_WEIGHT_MAX
    if concentration >= HYBRID_R_STOP:
        return HYBRID_INFORMATION_WEIGHT_MIN
    t = float((HYBRID_R_STOP - concentration) / (HYBRID_R_STOP - HYBRID_R_START))
    return float(HYBRID_INFORMATION_WEIGHT_MIN + (HYBRID_INFORMATION_WEIGHT_MAX - HYBRID_INFORMATION_WEIGHT_MIN) * t)


def hybrid_score_from_components(information_gain, helstrom_separation, max_information_gain, max_helstrom_separation, information_weight):
    information_norm = information_gain / max(max_information_gain, EPS)
    helstrom_norm = helstrom_separation / max(max_helstrom_separation, EPS)
    w = float(np.clip(information_weight, 0.0, 1.0))
    return float(w * information_norm + (1.0 - w) * helstrom_norm)


def readout_objective_for_state(shot, concentration):
    if READOUT_OBJECTIVE == "mixed":
        return "information_gain" if float(concentration) < MIXED_PHASE_CONCENTRATION else "helstrom"
    if READOUT_OBJECTIVE == "mixed_guarded":
        return "information_gain" if float(concentration) < MIXED_PHASE_CONCENTRATION else "guarded_helstrom"
    if READOUT_OBJECTIVE == "periodic_hybrid":
        return "information_gain" if int(shot) % PERIODIC_HYBRID_PERIOD == 0 else "helstrom"
    if READOUT_OBJECTIVE == "periodic_bayes_hybrid":
        return "information_gain" if int(shot) % PERIODIC_HYBRID_PERIOD == 0 else "bayes_drift"
    if READOUT_OBJECTIVE == "periodic_stein_hybrid":
        return "information_gain" if int(shot) % PERIODIC_HYBRID_PERIOD == 0 else "stein_drift"
    return READOUT_OBJECTIVE


def angle_objective(cfg, particle_grid, weights, h1_probability, readout_polar, readout_angle, start_time, objective, lambda_information):
    if objective == "helstrom":
        return angle_helstrom(cfg, particle_grid, weights, readout_polar, readout_angle, start_time)
    if objective == "bayes_drift":
        return angle_bayes_drift(cfg, particle_grid, weights, readout_polar, readout_angle, start_time)
    if objective == "stein_drift":
        return angle_stein_drift(cfg, particle_grid, weights, readout_polar, readout_angle, start_time)
    if objective == "blended":
        information_gain, particle_one, mixture_one = angle_utility(
            cfg, particle_grid, weights, h1_probability, readout_polar, readout_angle, start_time
        )
        bayes_drift = float(binary_kl(mixture_one, h0_probability_one_for_action(readout_polar, readout_angle, cfg)))
        return bayes_drift + lambda_information * information_gain, particle_one, mixture_one
    return angle_utility(cfg, particle_grid, weights, h1_probability, readout_polar, readout_angle, start_time)


def refine_continuous_angle(cfg, particle_grid, weights, h1_probability, center_angle, bracket_width, start_time, objective, lambda_information):
    left = center_angle - bracket_width
    right = center_angle + bracket_width
    best_angle = float(center_angle)
    best_utility, best_particle_one, best_mixture_one = angle_objective(
        cfg, particle_grid, weights, h1_probability, 0.5 * math.pi, best_angle, start_time, objective, lambda_information
    )

    for _ in range(8):
        mid_left = left + (right - left) / 3.0
        mid_right = right - (right - left) / 3.0
        utility_left, particle_one_left, mixture_left = angle_objective(
            cfg, particle_grid, weights, h1_probability, 0.5 * math.pi, mid_left, start_time, objective, lambda_information
        )
        utility_right, particle_one_right, mixture_right = angle_objective(
            cfg, particle_grid, weights, h1_probability, 0.5 * math.pi, mid_right, start_time, objective, lambda_information
        )
        if utility_left > best_utility + 1.0e-14:
            best_angle = float(mid_left)
            best_utility = utility_left
            best_particle_one = particle_one_left
            best_mixture_one = mixture_left
        if utility_right > best_utility + 1.0e-14:
            best_angle = float(mid_right)
            best_utility = utility_right
            best_particle_one = particle_one_right
            best_mixture_one = mixture_right
        if utility_left < utility_right:
            left = mid_left
        else:
            right = mid_right

    best_polar, best_angle = canonical_readout_action(0.5 * math.pi, best_angle)
    return best_polar, best_angle, best_utility, best_particle_one, best_mixture_one


def refine_continuous_sphere_action(cfg, particle_grid, weights, h1_probability, center_polar, center_angle, start_time, objective, lambda_information):
    best_polar, best_angle = canonical_readout_action(center_polar, center_angle)
    best_utility, best_particle_one, best_mixture_one = angle_objective(
        cfg, particle_grid, weights, h1_probability, best_polar, best_angle, start_time, objective, lambda_information
    )
    polar_step = math.pi if len(CANDIDATE_POLARS) <= 1 else math.pi / max(1, len(CANDIDATE_POLARS) - 1)
    angle_step = math.pi if len(cfg.candidate_angles) <= 1 else math.pi / len(cfg.candidate_angles)
    for _ in range(8):
        trial_actions = []
        for dp in (-polar_step, 0.0, polar_step):
            for da in (-angle_step, 0.0, angle_step):
                if dp == 0.0 and da == 0.0:
                    continue
                trial_actions.append(canonical_readout_action(best_polar + dp, best_angle + da))
        for trial_polar, trial_angle in trial_actions:
            utility, particle_one, mixture_one = angle_objective(
                cfg, particle_grid, weights, h1_probability, trial_polar, trial_angle, start_time, objective, lambda_information
            )
            if utility > best_utility + 1.0e-14:
                best_polar = float(trial_polar)
                best_angle = float(trial_angle)
                best_utility = utility
                best_particle_one = particle_one
                best_mixture_one = mixture_one
        polar_step *= 0.5
        angle_step *= 0.5
    return best_polar, best_angle, best_utility, best_particle_one, best_mixture_one


def choose_hybrid_angle(cfg, particle_grid, weights, h1_probability, start_time, information_weight):
    coarse = []
    max_information_gain = 0.0
    max_helstrom_separation = 0.0
    for polar in CANDIDATE_POLARS:
        for angle in cfg.candidate_angles:
            _, separation, information_gain, particle_one, mixture_one = angle_guarded_helstrom(
                cfg,
                particle_grid,
                weights,
                h1_probability,
                float(polar),
                float(angle),
                start_time,
                0.0,
            )
            max_information_gain = max(max_information_gain, information_gain)
            max_helstrom_separation = max(max_helstrom_separation, separation)
            coarse.append((float(polar), float(angle), separation, information_gain, particle_one, mixture_one))

    def score_item(item):
        return hybrid_score_from_components(item[3], item[2], max_information_gain, max_helstrom_separation, information_weight)

    best_polar, best_angle, best_separation, best_information_gain, best_particle_one, best_mixture_one = max(
        coarse, key=score_item
    )
    best_utility = score_item((best_polar, best_angle, best_separation, best_information_gain, best_particle_one, best_mixture_one))

    if READOUT_OPTIMIZATION != "continuous":
        best_polar, best_angle = canonical_readout_action(best_polar, best_angle)
        return best_polar, best_angle, best_utility, best_particle_one, best_mixture_one

    if READOUT_SPACE == "sphere":
        polar_step = math.pi if len(CANDIDATE_POLARS) <= 1 else math.pi / max(1, len(CANDIDATE_POLARS) - 1)
        angle_step = math.pi if len(cfg.candidate_angles) <= 1 else math.pi / len(cfg.candidate_angles)
        best_polar, best_angle = canonical_readout_action(best_polar, best_angle)
        for _ in range(8):
            for dp in (-polar_step, 0.0, polar_step):
                for da in (-angle_step, 0.0, angle_step):
                    if dp == 0.0 and da == 0.0:
                        continue
                    trial_polar, trial_angle = canonical_readout_action(best_polar + dp, best_angle + da)
                    _, separation, information_gain, particle_one, mixture_one = angle_guarded_helstrom(
                        cfg,
                        particle_grid,
                        weights,
                        h1_probability,
                        trial_polar,
                        trial_angle,
                        start_time,
                        0.0,
                    )
                    utility = hybrid_score_from_components(
                        information_gain,
                        separation,
                        max_information_gain,
                        max_helstrom_separation,
                        information_weight,
                    )
                    if utility > best_utility + 1.0e-14:
                        best_polar = float(trial_polar)
                        best_angle = float(trial_angle)
                        best_utility = utility
                        best_particle_one = particle_one
                        best_mixture_one = mixture_one
            polar_step *= 0.5
            angle_step *= 0.5
        return best_polar, best_angle, best_utility, best_particle_one, best_mixture_one

    bracket_width = math.pi if len(cfg.candidate_angles) <= 1 else math.pi / len(cfg.candidate_angles)
    left = best_angle - bracket_width
    right = best_angle + bracket_width
    best_polar, best_angle = canonical_readout_action(0.5 * math.pi, best_angle)
    for _ in range(8):
        mid_left = left + (right - left) / 3.0
        mid_right = right - (right - left) / 3.0
        _, separation_left, information_left, particle_one_left, mixture_left = angle_guarded_helstrom(
            cfg, particle_grid, weights, h1_probability, 0.5 * math.pi, mid_left, start_time, 0.0
        )
        _, separation_right, information_right, particle_one_right, mixture_right = angle_guarded_helstrom(
            cfg, particle_grid, weights, h1_probability, 0.5 * math.pi, mid_right, start_time, 0.0
        )
        utility_left = hybrid_score_from_components(
            information_left, separation_left, max_information_gain, max_helstrom_separation, information_weight
        )
        utility_right = hybrid_score_from_components(
            information_right, separation_right, max_information_gain, max_helstrom_separation, information_weight
        )
        if utility_left > best_utility + 1.0e-14:
            best_polar, best_angle = canonical_readout_action(0.5 * math.pi, mid_left)
            best_utility = utility_left
            best_particle_one = particle_one_left
            best_mixture_one = mixture_left
        if utility_right > best_utility + 1.0e-14:
            best_polar, best_angle = canonical_readout_action(0.5 * math.pi, mid_right)
            best_utility = utility_right
            best_particle_one = particle_one_right
            best_mixture_one = mixture_right
        if utility_left < utility_right:
            left = mid_left
        else:
            right = mid_right

    return best_polar, best_angle, best_utility, best_particle_one, best_mixture_one


def choose_guarded_helstrom_angle(cfg, particle_grid, weights, h1_probability, start_time):
    coarse = []
    max_information_gain = 0.0
    for polar in CANDIDATE_POLARS:
        for angle in cfg.candidate_angles:
            _, separation, information_gain, particle_one, mixture_one = angle_guarded_helstrom(
                cfg,
                particle_grid,
                weights,
                h1_probability,
                float(polar),
                float(angle),
                start_time,
                0.0,
            )
            max_information_gain = max(max_information_gain, information_gain)
            coarse.append((float(polar), float(angle), separation, information_gain, particle_one, mixture_one))

    information_threshold = max(GUARDED_INFORMATION_FLOOR, GUARDED_INFORMATION_FRACTION * max_information_gain)
    valid = [item for item in coarse if item[3] + 1.0e-14 >= information_threshold]
    if not valid:
        # If an absolute floor is set too high, fall back to the most informative action rather than returning a dead action.
        valid = [max(coarse, key=lambda item: item[3])]
        information_threshold = valid[0][3]
    best_polar, best_angle, best_utility, _, best_particle_one, best_mixture_one = max(valid, key=lambda item: item[2])

    if READOUT_OPTIMIZATION != "continuous":
        best_polar, best_angle = canonical_readout_action(best_polar, best_angle)
        return best_polar, best_angle, best_utility, best_particle_one, best_mixture_one

    if READOUT_SPACE == "sphere":
        polar_step = math.pi if len(CANDIDATE_POLARS) <= 1 else math.pi / max(1, len(CANDIDATE_POLARS) - 1)
        angle_step = math.pi if len(cfg.candidate_angles) <= 1 else math.pi / len(cfg.candidate_angles)
        best_polar, best_angle = canonical_readout_action(best_polar, best_angle)
        for _ in range(8):
            for dp in (-polar_step, 0.0, polar_step):
                for da in (-angle_step, 0.0, angle_step):
                    if dp == 0.0 and da == 0.0:
                        continue
                    trial_polar, trial_angle = canonical_readout_action(best_polar + dp, best_angle + da)
                    score, separation, _, particle_one, mixture_one = angle_guarded_helstrom(
                        cfg,
                        particle_grid,
                        weights,
                        h1_probability,
                        trial_polar,
                        trial_angle,
                        start_time,
                        information_threshold,
                    )
                    if score > best_utility + 1.0e-14:
                        best_polar = float(trial_polar)
                        best_angle = float(trial_angle)
                        best_utility = separation
                        best_particle_one = particle_one
                        best_mixture_one = mixture_one
            polar_step *= 0.5
            angle_step *= 0.5
        return best_polar, best_angle, best_utility, best_particle_one, best_mixture_one

    bracket_width = math.pi if len(cfg.candidate_angles) <= 1 else math.pi / len(cfg.candidate_angles)
    left = best_angle - bracket_width
    right = best_angle + bracket_width
    best_polar, best_angle = canonical_readout_action(0.5 * math.pi, best_angle)
    for _ in range(8):
        mid_left = left + (right - left) / 3.0
        mid_right = right - (right - left) / 3.0
        score_left, separation_left, _, particle_one_left, mixture_left = angle_guarded_helstrom(
            cfg, particle_grid, weights, h1_probability, 0.5 * math.pi, mid_left, start_time, information_threshold
        )
        score_right, separation_right, _, particle_one_right, mixture_right = angle_guarded_helstrom(
            cfg, particle_grid, weights, h1_probability, 0.5 * math.pi, mid_right, start_time, information_threshold
        )
        if score_left > best_utility + 1.0e-14:
            best_polar, best_angle = canonical_readout_action(0.5 * math.pi, mid_left)
            best_utility = separation_left
            best_particle_one = particle_one_left
            best_mixture_one = mixture_left
        if score_right > best_utility + 1.0e-14:
            best_polar, best_angle = canonical_readout_action(0.5 * math.pi, mid_right)
            best_utility = separation_right
            best_particle_one = particle_one_right
            best_mixture_one = mixture_right
        if score_left < score_right:
            left = mid_left
        else:
            right = mid_right

    return best_polar, best_angle, best_utility, best_particle_one, best_mixture_one


def choose_detection_angle_fast(cfg, particle_grid, weights, start_time, objective):
    moments = posterior_bloch_moments_for_detection(cfg, particle_grid, weights, start_time)
    best_polar = 0.5 * math.pi
    best_angle = float(cfg.candidate_angles[0])
    best_utility = -np.inf
    best_mixture_one = 0.5

    for polar in CANDIDATE_POLARS:
        for angle in cfg.candidate_angles:
            utility, mixture_one = detection_objective_from_moments(
                cfg, moments, float(polar), float(angle), objective
            )
            if utility > best_utility + 1.0e-14:
                best_polar = float(polar)
                best_angle = float(angle)
                best_utility = utility
                best_mixture_one = mixture_one

    if READOUT_OPTIMIZATION == "continuous":
        if READOUT_SPACE == "sphere":
            polar_step = math.pi if len(CANDIDATE_POLARS) <= 1 else math.pi / max(1, len(CANDIDATE_POLARS) - 1)
            angle_step = math.pi if len(cfg.candidate_angles) <= 1 else math.pi / len(cfg.candidate_angles)
            best_polar, best_angle = canonical_readout_action(best_polar, best_angle)
            for _ in range(8):
                for dp in (-polar_step, 0.0, polar_step):
                    for da in (-angle_step, 0.0, angle_step):
                        if dp == 0.0 and da == 0.0:
                            continue
                        trial_polar, trial_angle = canonical_readout_action(best_polar + dp, best_angle + da)
                        utility, mixture_one = detection_objective_from_moments(
                            cfg, moments, trial_polar, trial_angle, objective
                        )
                        if utility > best_utility + 1.0e-14:
                            best_polar = float(trial_polar)
                            best_angle = float(trial_angle)
                            best_utility = utility
                            best_mixture_one = mixture_one
                polar_step *= 0.5
                angle_step *= 0.5
        else:
            bracket_width = math.pi if len(cfg.candidate_angles) <= 1 else math.pi / len(cfg.candidate_angles)
            left = best_angle - bracket_width
            right = best_angle + bracket_width
            best_polar, best_angle = canonical_readout_action(0.5 * math.pi, best_angle)
            for _ in range(8):
                mid_left = left + (right - left) / 3.0
                mid_right = right - (right - left) / 3.0
                utility_left, mixture_left = detection_objective_from_moments(
                    cfg, moments, 0.5 * math.pi, mid_left, objective
                )
                utility_right, mixture_right = detection_objective_from_moments(
                    cfg, moments, 0.5 * math.pi, mid_right, objective
                )
                if utility_left > best_utility + 1.0e-14:
                    best_polar, best_angle = canonical_readout_action(0.5 * math.pi, mid_left)
                    best_utility = utility_left
                    best_mixture_one = mixture_left
                if utility_right > best_utility + 1.0e-14:
                    best_polar, best_angle = canonical_readout_action(0.5 * math.pi, mid_right)
                    best_utility = utility_right
                    best_mixture_one = mixture_right
                if utility_left < utility_right:
                    left = mid_left
                else:
                    right = mid_right

    best_polar, best_angle = canonical_readout_action(best_polar, best_angle)
    particle_one = particle_probabilities_for_action(cfg, particle_grid, best_polar, best_angle, start_time)
    best_mixture_one = float(np.dot(weights, particle_one))
    return best_polar, best_angle, best_utility, particle_one, best_mixture_one


def choose_adaptive_angle(cfg, particle_grid, weights, h1_probability, start_time, objective, lambda_information):
    if FAST_DETECTION_OBJECTIVES and objective in {"helstrom", "bayes_drift", "stein_drift"}:
        return choose_detection_angle_fast(cfg, particle_grid, weights, start_time, objective)

    if objective == "hybrid":
        return choose_hybrid_angle(cfg, particle_grid, weights, h1_probability, start_time, lambda_information)

    if objective == "guarded_helstrom":
        return choose_guarded_helstrom_angle(cfg, particle_grid, weights, h1_probability, start_time)

    best_polar = 0.5 * math.pi
    best_angle = float(cfg.candidate_angles[0])
    best_utility = -np.inf
    best_particle_one = None
    best_mixture_one = 0.5

    for polar in CANDIDATE_POLARS:
        for angle in cfg.candidate_angles:
            utility, particle_one, mixture_one = angle_objective(
                cfg, particle_grid, weights, h1_probability, float(polar), float(angle), start_time, objective, lambda_information
            )
            if utility > best_utility + 1.0e-14:
                best_polar = float(polar)
                best_angle = float(angle)
                best_utility = utility
                best_particle_one = particle_one
                best_mixture_one = mixture_one

    if READOUT_OPTIMIZATION == "continuous":
        if READOUT_SPACE == "sphere":
            return refine_continuous_sphere_action(
                cfg, particle_grid, weights, h1_probability, best_polar, best_angle, start_time, objective, lambda_information
            )
        bracket_width = math.pi if len(cfg.candidate_angles) <= 1 else math.pi / len(cfg.candidate_angles)
        return refine_continuous_angle(
            cfg, particle_grid, weights, h1_probability, best_angle, bracket_width, start_time, objective, lambda_information
        )

    return best_polar, best_angle, best_utility, best_particle_one, best_mixture_one


def snapshot_payload(particle_grid, weights, h1_probability, log_bayes_factor):
    conditional = np.asarray(weights, dtype=float).reshape(particle_grid.shape).sum(axis=1)
    conditional = conditional / conditional.sum()
    return {
        "conditional": conditional,
        "h1_probability": float(h1_probability),
        "log_bayes_factor": float(log_bayes_factor),
        "amplitude_marginal": conditional.sum(axis=1),
        "phase_marginal": conditional.sum(axis=0),
    }


def run_adaptive_posterior_trajectory(cfg, hypothesis, snapshot_shots, seed):
    if hypothesis not in {"H0", "H1"}:
        raise ValueError("hypothesis must be H0 or H1")
    particle_grid = build_rabi_particle_grid(cfg.particles)
    rng = np.random.default_rng(seed)
    weights = np.asarray(particle_grid.prior_weights, dtype=float).copy()
    h1_probability = float(cfg.initial_h1_probability)
    log_bayes_factor = 0.0
    current_time = 0.0
    snapshot_set = set(int(s) for s in snapshot_shots)
    snapshots = {}
    records = []

    if 0 in snapshot_set:
        snapshots[0] = snapshot_payload(particle_grid, weights, h1_probability, log_bayes_factor)

    for shot in range(int(cfg.n_shots)):
        current_phase_concentration = phase_concentration(particle_grid, weights)
        active_objective = readout_objective_for_state(shot, current_phase_concentration)
        if active_objective == "hybrid":
            lambda_information = hybrid_information_weight_from_concentration(current_phase_concentration)
        elif active_objective == "blended":
            lambda_information = blend_lambda_from_concentration(current_phase_concentration)
        else:
            lambda_information = 0.0
        readout_polar, readout_angle, utility, particle_one, mixture_one = choose_adaptive_angle(
            cfg,
            particle_grid,
            weights,
            h1_probability,
            current_time,
            active_objective,
            lambda_information,
        )
        h0_one = h0_probability_one_for_action(readout_polar, readout_angle, cfg)
        if hypothesis == "H1":
            true_one = true_probability_one_for_action(cfg, readout_polar, readout_angle, current_time)
        else:
            true_one = h0_one
        outcome = bool(rng.random() < true_one)
        particle_likelihood = particle_one if outcome else (1.0 - particle_one)
        mixture_likelihood = float(np.clip(np.dot(weights, particle_likelihood), EPS, 1.0 - EPS))
        h0_likelihood = float(np.clip(h0_one if outcome else (1.0 - h0_one), EPS, 1.0 - EPS))
        weights = weights * particle_likelihood / mixture_likelihood
        weights = weights / weights.sum()

        evidence = h1_probability * mixture_likelihood + (1.0 - h1_probability) * h0_likelihood
        evidence = float(np.clip(evidence, EPS, 1.0))
        h1_probability = h1_probability * mixture_likelihood / evidence
        log_bayes_factor += math.log(mixture_likelihood) - math.log(h0_likelihood)

        records.append(
            {
                "shot": shot + 1,
                "h1_probability": h1_probability,
                "log_bayes_factor": log_bayes_factor,
                "readout_angle": readout_angle,
                "readout_polar_angle": readout_polar,
                "active_readout_objective": active_objective,
                "phase_concentration": current_phase_concentration,
                "blend_lambda": lambda_information,
                "readout_objective_value": utility,
                "expected_information_gain": utility if active_objective == "information_gain" else np.nan,
                "expected_helstrom_separation": utility if active_objective in {"helstrom", "guarded_helstrom"} else np.nan,
                "expected_bayes_drift": utility if active_objective == "bayes_drift" else np.nan,
                "expected_stein_drift": utility if active_objective == "stein_drift" else np.nan,
                "expected_hybrid_score": utility if active_objective == "hybrid" else np.nan,
                "hybrid_information_weight": lambda_information if active_objective == "hybrid" else np.nan,
                "outcome": int(outcome),
                "predictive_h1_one": mixture_one,
                "true_one": true_one,
            }
        )

        current_time += cfg.sensing_time + cfg.dead_time
        if shot + 1 in snapshot_set:
            snapshots[shot + 1] = snapshot_payload(particle_grid, weights, h1_probability, log_bayes_factor)

    return {"grid": particle_grid, "records": pd.DataFrame(records), "snapshots": snapshots}


In [ ]:

def stable_hash(payload: dict) -> str:
    blob = json.dumps(payload, sort_keys=True, default=list).encode("utf-8")
    return hashlib.sha256(blob).hexdigest()[:16]


def trajectory_cache_path(hypothesis: str, seed: int) -> Path:
    payload = setup | {"hypothesis": hypothesis, "seed": int(seed)}
    return CACHE_DIR / f"{OUTPUT_BASENAME}_{OUTPUT_TAG}_{stable_hash(payload)}_{hypothesis}.npz"


def compatible_legacy_high_fidelity_setup() -> dict | None:
    if DETECTOR_PROFILE != "high_fidelity" or RUN_PRESET != "final":
        return None
    if not LEGACY_HIGH_FIDELITY_SETUP_PATH.exists():
        return None

    legacy_setup = json.loads(LEGACY_HIGH_FIDELITY_SETUP_PATH.read_text(encoding="utf-8"))
    for key, legacy_value in legacy_setup.items():
        if key == "cache_version":
            continue
        if key not in setup:
            return None
        current_blob = json.dumps(setup[key], sort_keys=True, default=list)
        legacy_blob = json.dumps(legacy_value, sort_keys=True, default=list)
        if current_blob != legacy_blob:
            return None
    return legacy_setup


def legacy_high_fidelity_cache_path(payload_updates: dict, filename_suffix: str) -> Path | None:
    legacy_setup = compatible_legacy_high_fidelity_setup()
    if legacy_setup is None:
        return None
    payload = legacy_setup | payload_updates
    legacy_hash = stable_hash(payload)
    path = CACHE_DIR / (
        f"{OUTPUT_BASENAME}_{legacy_setup['run_preset']}_{legacy_hash}_{filename_suffix}.npz"
    )
    return path if path.exists() else None


def save_trajectory(path: Path, result: dict) -> None:
    snapshot_shots = np.array(sorted(result["snapshots"]), dtype=int)
    conditional_stack = np.stack([result["snapshots"][int(s)]["conditional"] for s in snapshot_shots], axis=0)
    h1_probability = np.array([result["snapshots"][int(s)]["h1_probability"] for s in snapshot_shots], dtype=float)
    log_bayes_factor = np.array([result["snapshots"][int(s)]["log_bayes_factor"] for s in snapshot_shots], dtype=float)
    np.savez_compressed(
        path,
        records=result["records"].to_records(index=False),
        snapshot_shots=snapshot_shots,
        conditional_stack=conditional_stack,
        h1_probability=h1_probability,
        log_bayes_factor=log_bayes_factor,
    )


def load_trajectory(path: Path, particle_grid) -> dict:
    payload = np.load(path, allow_pickle=True)
    records = pd.DataFrame.from_records(payload["records"])
    snapshots = {}
    for index, shot in enumerate(payload["snapshot_shots"]):
        conditional = np.asarray(payload["conditional_stack"][index], dtype=float)
        snapshots[int(shot)] = {
            "conditional": conditional,
            "h1_probability": float(payload["h1_probability"][index]),
            "log_bayes_factor": float(payload["log_bayes_factor"][index]),
            "amplitude_marginal": conditional.sum(axis=1),
            "phase_marginal": conditional.sum(axis=0),
        }
    return {"grid": particle_grid, "records": records, "snapshots": snapshots}


def source_trajectory_to_result(raw: dict, particle_grid) -> dict:
    columns = [
        "shot",
        "h1_probability",
        "log_bayes_factor",
        "readout_angle",
        "readout_polar_angle",
        "active_objective_code",
        "phase_concentration",
        "readout_objective_value",
        "expected_information_gain",
        "expected_helstrom_separation",
        "expected_bayes_drift",
        "expected_stein_drift",
        "outcome",
        "predictive_h1_one",
        "true_one",
    ]
    records = pd.DataFrame(np.asarray(raw["records"], dtype=float), columns=columns)
    records["shot"] = records["shot"].astype(int)
    records["outcome"] = records["outcome"].astype(int)
    objective_labels = {0: "information_gain", 1: "helstrom", 2: "bayes_drift", 3: "stein_drift"}
    records["active_readout_objective"] = [objective_labels[int(code)] for code in records.pop("active_objective_code")]
    records["blend_lambda"] = np.nan
    records["expected_hybrid_score"] = np.nan
    records["hybrid_information_weight"] = np.nan

    snapshots = {}
    for index, shot in enumerate(np.asarray(raw["snapshot_shots"], dtype=int)):
        weights = np.asarray(raw["snapshot_weights"][index], dtype=float)
        conditional = weights.reshape(particle_grid.shape).sum(axis=1)
        conditional = conditional / conditional.sum()
        snapshots[int(shot)] = {
            "conditional": conditional,
            "h1_probability": float(raw["snapshot_h1_probabilities"][index]),
            "log_bayes_factor": float(raw["snapshot_log_bayes_factors"][index]),
            "amplitude_marginal": conditional.sum(axis=1),
            "phase_marginal": conditional.sum(axis=0),
        }
    return {"grid": particle_grid, "records": records, "snapshots": snapshots}


def run_source_posterior_trajectory(cfg, hypothesis: str, snapshot_shots, seed: int) -> dict:
    if not USE_SOURCE_TRAJECTORY:
        raise RuntimeError("source trajectory backend is not active")
    raw = run_rabi_sphere_learning_trajectory_numba(
        cfg,
        SOURCE_TRAJECTORY_STRATEGY,
        hypothesis,
        snapshot_shots,
        seed=seed,
    )
    return source_trajectory_to_result(raw, build_rabi_particle_grid(cfg.particles))


LOADED_TRAJECTORY_CACHE_PATHS: dict[str, Path] = {}


def cached_trajectory(hypothesis: str, seed: int) -> dict:
    path = trajectory_cache_path(hypothesis, seed)
    particle_grid = build_rabi_particle_grid(config.particles)
    if USE_CACHE and path.exists() and not FORCE_RECOMPUTE:
        LOADED_TRAJECTORY_CACHE_PATHS[hypothesis] = path
        print(f"loaded {hypothesis} trajectory from {path.name}")
        return load_trajectory(path, particle_grid)
    legacy_path = (
        legacy_high_fidelity_cache_path(
            {"hypothesis": hypothesis, "seed": int(seed)},
            hypothesis,
        )
        if USE_CACHE
        else None
    )
    if legacy_path is not None and not FORCE_RECOMPUTE:
        LOADED_TRAJECTORY_CACHE_PATHS[hypothesis] = legacy_path
        print(f"loaded {hypothesis} trajectory from legacy cache {legacy_path.name}")
        return load_trajectory(legacy_path, particle_grid)
    if os.environ.get("RABI_ALLOW_SIMULATION", "0") != "1":
        raise FileNotFoundError(f"Missing cached trajectory: {path}. Set RABI_ALLOW_SIMULATION=1 to simulate.")
    if USE_SOURCE_TRAJECTORY:
        result = run_source_posterior_trajectory(config, hypothesis, SNAPSHOT_SHOTS, seed)
    else:
        result = run_adaptive_posterior_trajectory(config, hypothesis, SNAPSHOT_SHOTS, seed)
    if USE_CACHE:
        save_trajectory(path, result)
    LOADED_TRAJECTORY_CACHE_PATHS[hypothesis] = path
    return result


def mixed_switch_shot(records):
    if "active_readout_objective" not in records.columns:
        return None
    helstrom_rows = records[records["active_readout_objective"].isin(["helstrom", "guarded_helstrom"])]
    if helstrom_rows.empty:
        return None
    return int(helstrom_rows.iloc[0]["shot"])


def draw_mixed_switch_line(ax, shot, color, label=None):
    if shot is None:
        return
    ax.axvline(shot, color=color, lw=1.2, ls="--", alpha=0.85, zorder=1)
    if label is not None:
        ax.text(shot, 0.98, label, transform=ax.get_xaxis_transform(), rotation=90, va="top", ha="right", fontsize=8, color=color)


FIXED_TRAJECTORY_SPECS = {
    "fixed_z": {"label": "fixed z"},
    "fixed_x": {"label": "fixed x"},
    "fixed_xy": {"label": "fixed x/y"},
}


def fixed_readout_action(strategy: str, shot: int) -> tuple[float, float]:
    if strategy == "fixed_z":
        return 0.0, 0.0
    if strategy == "fixed_x":
        return 0.5 * math.pi, 0.0
    if strategy == "fixed_xy":
        return 0.5 * math.pi, 0.5 * math.pi * (int(shot) % 2)
    raise ValueError(f"unknown fixed readout strategy: {strategy}")


def run_fixed_posterior_trajectory(cfg, hypothesis, snapshot_shots, seed, strategy):
    if hypothesis not in {"H0", "H1"}:
        raise ValueError("hypothesis must be H0 or H1")
    particle_grid = build_rabi_particle_grid(cfg.particles)
    rng = np.random.default_rng(seed)
    weights = np.asarray(particle_grid.prior_weights, dtype=float).copy()
    h1_probability = float(cfg.initial_h1_probability)
    log_bayes_factor = 0.0
    current_time = 0.0
    snapshot_set = set(int(s) for s in snapshot_shots)
    snapshots = {}
    records = []

    if 0 in snapshot_set:
        snapshots[0] = snapshot_payload(particle_grid, weights, h1_probability, log_bayes_factor)

    for shot in range(int(cfg.n_shots)):
        readout_polar, readout_angle = fixed_readout_action(strategy, shot)
        particle_one = particle_probabilities_for_action(cfg, particle_grid, readout_polar, readout_angle, current_time)
        mixture_one = float(np.clip(np.dot(weights, particle_one), EPS, 1.0 - EPS))
        h0_one = h0_probability_one_for_action(readout_polar, readout_angle, cfg)
        if hypothesis == "H1":
            true_one = true_probability_one_for_action(cfg, readout_polar, readout_angle, current_time)
        else:
            true_one = h0_one
        outcome = bool(rng.random() < true_one)
        particle_likelihood = particle_one if outcome else (1.0 - particle_one)
        mixture_likelihood = float(np.clip(np.dot(weights, particle_likelihood), EPS, 1.0 - EPS))
        h0_likelihood = float(np.clip(h0_one if outcome else (1.0 - h0_one), EPS, 1.0 - EPS))
        weights = weights * particle_likelihood / mixture_likelihood
        weights = weights / weights.sum()

        evidence = h1_probability * mixture_likelihood + (1.0 - h1_probability) * h0_likelihood
        evidence = float(np.clip(evidence, EPS, 1.0))
        h1_probability = h1_probability * mixture_likelihood / evidence
        log_bayes_factor += math.log(mixture_likelihood) - math.log(h0_likelihood)

        records.append(
            {
                "shot": shot + 1,
                "h1_probability": h1_probability,
                "log_bayes_factor": log_bayes_factor,
                "readout_angle": readout_angle,
                "readout_polar_angle": readout_polar,
                "active_readout_objective": strategy,
                "phase_concentration": phase_concentration(particle_grid, weights),
                "blend_lambda": np.nan,
                "readout_objective_value": np.nan,
                "expected_information_gain": np.nan,
                "expected_helstrom_separation": np.nan,
                "expected_bayes_drift": np.nan,
                "expected_stein_drift": np.nan,
                "expected_hybrid_score": np.nan,
                "hybrid_information_weight": np.nan,
                "outcome": int(outcome),
                "predictive_h1_one": mixture_one,
                "true_one": true_one,
            }
        )

        current_time += cfg.sensing_time + cfg.dead_time
        if shot + 1 in snapshot_set:
            snapshots[shot + 1] = snapshot_payload(particle_grid, weights, h1_probability, log_bayes_factor)

    return {"grid": particle_grid, "records": pd.DataFrame(records), "snapshots": snapshots}


def fixed_trajectory_cache_path(strategy: str, hypothesis: str, seed: int) -> Path:
    payload = setup | {"hypothesis": hypothesis, "seed": int(seed), "trajectory_kind": "fixed_axis", "fixed_strategy": strategy}
    return CACHE_DIR / f"{OUTPUT_BASENAME}_{OUTPUT_TAG}_{stable_hash(payload)}_{strategy}_{hypothesis}.npz"


LOADED_FIXED_CACHE_PATHS: dict[str, Path] = {}


def cached_fixed_trajectory(strategy: str, hypothesis: str, seed: int) -> dict:
    path = fixed_trajectory_cache_path(strategy, hypothesis, seed)
    particle_grid = build_rabi_particle_grid(config.particles)
    if USE_CACHE and path.exists() and not FORCE_RECOMPUTE:
        LOADED_FIXED_CACHE_PATHS[strategy] = path
        print(f"loaded {strategy} {hypothesis} trajectory from {path.name}")
        return load_trajectory(path, particle_grid)
    legacy_path = (
        legacy_high_fidelity_cache_path(
            {
                "hypothesis": hypothesis,
                "seed": int(seed),
                "trajectory_kind": "fixed_axis",
                "fixed_strategy": strategy,
            },
            f"{strategy}_{hypothesis}",
        )
        if USE_CACHE
        else None
    )
    if legacy_path is not None and not FORCE_RECOMPUTE:
        LOADED_FIXED_CACHE_PATHS[strategy] = legacy_path
        print(f"loaded {strategy} {hypothesis} trajectory from legacy cache {legacy_path.name}")
        return load_trajectory(legacy_path, particle_grid)
    if os.environ.get("RABI_ALLOW_SIMULATION", "0") != "1":
        raise FileNotFoundError(f"Missing cached trajectory: {path}. Set RABI_ALLOW_SIMULATION=1 to simulate.")
    result = run_fixed_posterior_trajectory(config, hypothesis, SNAPSHOT_SHOTS, seed, strategy)
    if USE_CACHE:
        save_trajectory(path, result)
    LOADED_FIXED_CACHE_PATHS[strategy] = path
    return result


signal_learning = cached_trajectory("H1", SEED_SIGNAL)
background_learning = cached_trajectory("H0", SEED_BACKGROUND)
print("signal final:", signal_learning["records"].iloc[-1][["h1_probability", "log_bayes_factor"]].to_dict())
print("background final:", background_learning["records"].iloc[-1][["h1_probability", "log_bayes_factor"]].to_dict())
print("cache sources:", LOADED_TRAJECTORY_CACHE_PATHS["H1"].name, LOADED_TRAJECTORY_CACHE_PATHS["H0"].name)

fixed_signal_learning = {
    strategy: cached_fixed_trajectory(strategy, "H1", SEED_SIGNAL)
    for strategy in FIXED_TRAJECTORY_SPECS
}
for strategy, result in fixed_signal_learning.items():
    label = FIXED_TRAJECTORY_SPECS[strategy]["label"]
    print(f"{label} signal final:", result["records"].iloc[-1][["h1_probability", "log_bayes_factor"]].to_dict())
print("fixed signal cache sources:", ", ".join(LOADED_FIXED_CACHE_PATHS[strategy].name for strategy in FIXED_TRAJECTORY_SPECS))


In [ ]:
# Paper-style learning curves: two horizontal panels stacked for a one-column figure.
fig, axes = plt.subplots(2, 1, figsize=(6.4, 4.8), sharex=True, constrained_layout=True)
# Font sizes scale with figure width; see rabi_paper.mplstyle.
paper_width_pt = 72.0 * fig.get_figwidth()
trajectory_specs = [
    (r"signal", signal_learning, "#9A4D55"),
    (r"background", background_learning, "#536B78"),
]
for label, result, color in trajectory_specs:
    records = result["records"]
    axes[0].plot(records["shot"], records["h1_probability"], color=color, lw=2.2, label=label)
    axes[1].plot(records["shot"], records["log_bayes_factor"], color=color, lw=2.2, label=label)
    if READOUT_OBJECTIVE in {"mixed", "mixed_guarded"}:
        switch = mixed_switch_shot(records)
        for ax in axes:
            draw_mixed_switch_line(ax, switch, color, label="switch" if label == "signal" else None)

for ax in axes:
    ax.set_xscale("log")
    ax.xaxis.set_major_locator(LogLocator(base=10.0))
    ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.tick_params(axis="both", which="major", labelsize=0.030 * paper_width_pt, direction="in", length=6.0, width=1.0, top=True, right=True)
    ax.tick_params(axis="both", which="minor", direction="in", length=3.2, width=0.8, top=True, right=True)
    ax.grid(False, which="both")
axes[-1].set_xlim(1.0, 1.15 * N_SHOTS)
axes[0].set_ylabel(r"$q_k$", fontsize=0.035 * paper_width_pt)
axes[0].set_ylim(-0.02, 1.02)
axes[0].legend(frameon=False, fontsize=0.028 * paper_width_pt, loc="best", handlelength=2.8)
axes[1].set_ylabel(r"$\log B_{10,k}$", fontsize=0.035 * paper_width_pt)
axes[1].set_xlabel(r"shot label $k$", fontsize=0.035 * paper_width_pt)

pdf_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{OUTPUT_TAG}_learning_curves.pdf"
fig.savefig(pdf_path, bbox_inches="tight")
print(f"saved {pdf_path}")
display(fig)
plt.close(fig)


def posterior_edges_for_grid(particle_grid):
    phase_values = particle_grid.drive_phase_values
    amplitude_values = particle_grid.phase_amplitude_values
    phase_edges = np.linspace(0.0, 2.0 * math.pi, phase_values.size + 1)
    if amplitude_values.size == 1:
        amplitude_edges = np.array([0.5 * amplitude_values[0], 1.5 * amplitude_values[0]])
    else:
        log_centers = np.log(amplitude_values)
        log_edges = np.empty(amplitude_values.size + 1)
        log_edges[1:-1] = 0.5 * (log_centers[:-1] + log_centers[1:])
        log_edges[0] = log_centers[0] - 0.5 * (log_centers[1] - log_centers[0])
        log_edges[-1] = log_centers[-1] + 0.5 * (log_centers[-1] - log_centers[-2])
        amplitude_edges = np.exp(log_edges)
    return phase_edges, amplitude_edges


def plot_posterior_heatmaps(result, title, output_suffix, show_truth=True, color_vmax=None):
    selected_snapshots = [s for s in SNAPSHOT_SHOTS if s in result["snapshots"]]
    phase_edges, amplitude_edges = posterior_edges_for_grid(result["grid"])
    fig, axes = plt.subplots(1, len(selected_snapshots), figsize=(3.4 * len(selected_snapshots), 3.6), sharey=True)
    if len(selected_snapshots) == 1:
        axes = [axes]

    floor = 1.0e-8
    if color_vmax is None:
        color_vmax = max(
            floor * 10.0,
            max(float(np.max(result["snapshots"][shot]["conditional"])) for shot in selected_snapshots),
        )
    mesh = None
    for ax, shot in zip(axes, selected_snapshots):
        posterior = np.maximum(result["snapshots"][shot]["conditional"], floor)
        mesh = ax.pcolormesh(
            phase_edges,
            amplitude_edges,
            posterior,
            shading="auto",
            edgecolors="none",
            linewidth=0.0,
            antialiased=False,
            rasterized=True,
            norm=LogNorm(vmin=floor, vmax=color_vmax),
            cmap="mako" if "mako" in plt.colormaps() else "viridis",
        )
        if show_truth:
            ax.axvline(TRUE_DRIVE_PHASE % (2.0 * math.pi), color="white", lw=1.2, ls="--")
            ax.axhline(PHI_R_TRUE, color="white", lw=1.2, ls=":")
        h1p = result["snapshots"][shot]["h1_probability"]
        ax.set_title(rf"$k={shot}$" + "\n" + rf"$q_k={h1p:.2f}$")
        ax.set_xlabel(r"drive phase $\tilde{\varphi}$")
        ax.set_yscale("log")
        ax.set_box_aspect(0.62)
        ax.grid(False, which="both")
    axes[0].set_ylabel(r"phase amplitude $\Phi_R$")
    fig.colorbar(mesh, ax=axes, shrink=0.86, label=r"$\pi_k(\Phi_R,\tilde{\varphi})$")
    fig.suptitle(title, y=1.05)

    pdf_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{OUTPUT_TAG}_{output_suffix}.pdf"
    fig.savefig(pdf_path, bbox_inches="tight")
    print(f"saved {pdf_path}")
    return fig


def plot_posterior_heatmaps_paper_2x2(
    result,
    output_suffix,
    policy_label,
    shots=(0, 256, 1024, 4096),
    show_truth=True,
    color_vmax=None,
):
    selected_snapshots = [int(s) for s in shots if int(s) in result["snapshots"]]
    missing = [int(s) for s in shots if int(s) not in result["snapshots"]]
    if missing:
        raise ValueError(
            "requested paper heatmap snapshots are missing: "
            f"{missing}. Add them to RABI_LEARNING_SNAPSHOT_SHOTS and rerun/load the trajectory."
        )
    if len(selected_snapshots) != 4:
        raise ValueError("the paper heatmap layout expects exactly four snapshots")

    phase_edges, amplitude_edges = posterior_edges_for_grid(result["grid"])
    fig, axes = plt.subplots(2, 2, figsize=(8.4, 5.4), sharex=True, sharey=True)
    paper_width_pt = 72.0 * fig.get_figwidth()
    axes_flat = axes.ravel()
    floor = 1.0e-8
    if color_vmax is None:
        color_vmax = max(
            floor * 10.0,
            max(float(np.max(result["snapshots"][shot]["conditional"])) for shot in selected_snapshots),
        )
    mesh = None
    for ax, shot in zip(axes_flat, selected_snapshots):
        posterior = np.maximum(result["snapshots"][shot]["conditional"], floor)
        mesh = ax.pcolormesh(
            phase_edges,
            amplitude_edges,
            posterior,
            shading="auto",
            edgecolors="none",
            linewidth=0.0,
            antialiased=False,
            rasterized=True,
            norm=LogNorm(vmin=floor, vmax=color_vmax),
            cmap="mako" if "mako" in plt.colormaps() else "viridis",
        )
        if show_truth:
            ax.axvline(TRUE_DRIVE_PHASE % (2.0 * math.pi), color="white", lw=1.2, ls="--")
            ax.axhline(PHI_R_TRUE, color="white", lw=1.2, ls=":")
        h1p = result["snapshots"][shot]["h1_probability"]
        ax.set_title(rf"$k={shot}$,  $q_k={h1p:.2f}$", pad=5.0, fontsize=0.029 * paper_width_pt)
        ax.set_yscale("log")
        ax.set_box_aspect(0.62)
        ax.grid(False, which="both")
    fig.suptitle(rf"Policy: {policy_label}", y=1.015, fontsize=0.032 * paper_width_pt)
    fig.supxlabel(r"drive phase $\tilde{\varphi}$", y=0.025, fontsize=0.035 * paper_width_pt)
    fig.supylabel(r"$\Phi_R$", x=0.015, fontsize=0.035 * paper_width_pt)
    fig.subplots_adjust(left=0.09, right=0.87, bottom=0.15, top=0.88, wspace=0.12, hspace=0.42)
    for ax in axes_flat:
        ax.tick_params(length=3.0, pad=2.0, labelsize=0.029 * paper_width_pt)
    cbar_ax = fig.add_axes([0.89, 0.19, 0.026, 0.62])
    cbar = fig.colorbar(mesh, cax=cbar_ax)
    cbar.set_label(r"$\pi_k(\Phi_R,\tilde{\varphi})$", fontsize=0.032 * paper_width_pt)
    cbar.ax.tick_params(labelsize=0.029 * paper_width_pt)

    pdf_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{OUTPUT_TAG}_{output_suffix}.pdf"
    fig.savefig(pdf_path, bbox_inches="tight")
    print(f"saved {pdf_path}")
    return fig


fig_ref = plot_posterior_heatmaps(
    background_learning,
    r"$H_1$-conditional posterior snapshots for one adaptive $H_0$ trajectory",
    "heatmaps_background",
    show_truth=False,
)
plt.close(fig_ref)


# The four signal-policy comparisons are drawn together in the next cell.


In [ ]:
# Four H1 signal policies.  Each policy gets both the full horizontal reference
# view and a separate manuscript-ready 2x2 view.
POLICY_HEATMAP_SPECS = [
    ("adaptive", "adaptive", signal_learning),
    ("fixed_z", r"fixed $z$", fixed_signal_learning["fixed_z"]),
    ("fixed_x", r"fixed $x$", fixed_signal_learning["fixed_x"]),
    ("fixed_xy", r"fixed $x/y$", fixed_signal_learning["fixed_xy"]),
]

# All policies start from exactly the same H1 prior, and use one shared color scale.
policy_prior = POLICY_HEATMAP_SPECS[0][2]["snapshots"][0]["conditional"]
for policy_name, _, result in POLICY_HEATMAP_SPECS[1:]:
    np.testing.assert_allclose(
        result["snapshots"][0]["conditional"],
        policy_prior,
        rtol=0.0,
        atol=1.0e-15,
        err_msg=f"k=0 prior differs for policy {policy_name}",
    )
POLICY_HEATMAP_VMAX = max(
    1.0e-7,
    max(
        float(np.max(snapshot["conditional"]))
        for _, _, result in POLICY_HEATMAP_SPECS
        for snapshot in result["snapshots"].values()
    ),
)

for policy_name, policy_label, result in POLICY_HEATMAP_SPECS:
    fig_ref = plot_posterior_heatmaps(
        result,
        rf"$H_1$-conditional posterior snapshots: {policy_label}",
        f"heatmaps_{policy_name}",
        color_vmax=POLICY_HEATMAP_VMAX,
    )
    display(fig_ref)
    plt.close(fig_ref)

PAPER_HEATMAP_SHOTS = tuple(
    int(x)
    for x in os.environ.get("RABI_LEARNING_PAPER_HEATMAP_SHOTS", "0,256,1024,4096").split(",")
    if x.strip()
)
for policy_name, policy_label, result in POLICY_HEATMAP_SPECS:
    fig_paper = plot_posterior_heatmaps_paper_2x2(
        result,
        f"heatmaps_{policy_name}_paper_2x2",
        policy_label,
        shots=PAPER_HEATMAP_SHOTS,
        color_vmax=POLICY_HEATMAP_VMAX,
    )
    display(fig_paper)
    plt.close(fig_paper)


In [ ]:

def ideal_signal_polar_angle(cfg):
    h0_z = probability_to_bloch_projection(h0_probability_one("z", 0.0, cfg), cfg.readout)
    true_z = probability_to_bloch_projection(
        rabi_probability_one(
            cfg.true_phase_amplitude,
            cfg.true_detuning,
            cfg.true_drive_phase,
            0.0,
            cfg.sensing_time,
            "z",
            0.0,
            cfg.readout,
        ),
        cfg.readout,
    )
    true_transverse = probability_to_bloch_projection(
        rabi_probability_one(
            cfg.true_phase_amplitude,
            cfg.true_detuning,
            cfg.true_drive_phase,
            0.0,
            cfg.sensing_time,
            "transverse",
            cfg.true_drive_phase + 0.5 * math.pi,
            cfg.readout,
        ),
        cfg.readout,
    )
    delta_z = float(true_z - h0_z)
    delta_transverse = float(abs(true_transverse))
    if delta_z == 0.0 and delta_transverse == 0.0:
        return 0.5 * math.pi
    return float(math.atan2(delta_transverse, delta_z))


IDEAL_SIGNAL_POLAR_ANGLE = ideal_signal_polar_angle(config)

records = signal_learning["records"]
READOUT_TRAJECTORY_MAX_SHOT = int(os.environ.get("RABI_LEARNING_READOUT_MAX_SHOT", str(N_SHOTS)))
plot_records = records[records["shot"] <= min(N_SHOTS, READOUT_TRAJECTORY_MAX_SHOT)]
signal_switch_shot = mixed_switch_shot(records) if READOUT_OBJECTIVE in {"mixed", "mixed_guarded"} else None
has_polar = "readout_polar_angle" in plot_records.columns
if has_polar:
    fig, axes = plt.subplots(2, 1, figsize=(8.2, 5.0), sharex=True)
    ax_angle, ax_polar = axes
else:
    fig, ax_angle = plt.subplots(figsize=(8.2, 3.2))
    ax_polar = None
# Scale type to the full exported figure, not to each small panel.
paper_width_pt = 72.0 * fig.get_figwidth()
OBJECTIVE_STYLE = {
    "information_gain": dict(color="#C14953", label="information gain", marker="o", alpha=0.62),
    "helstrom": dict(color="#2A6FBB", label="Helstrom", marker="o", alpha=0.48),
    "guarded_helstrom": dict(color="#3E8F6B", label="guarded Helstrom", marker="o", alpha=0.54),
    "bayes_drift": dict(color="#8A5A9E", label="Bayes drift", marker="o", alpha=0.50),
    "stein_drift": dict(color="#2A6FBB", label="KL / Stein drift", marker="o", alpha=0.48),
    "hybrid": dict(color="#D18F2F", label="hybrid", marker="o", alpha=0.52),
}


def scatter_readout_by_objective(ax, y_values, ylabel=None):
    if "active_readout_objective" not in plot_records.columns:
        ax.scatter(plot_records["shot"], y_values, s=8, alpha=0.45, color="#9A4D55")
        return
    for objective, group in plot_records.groupby("active_readout_objective", sort=False):
        style = OBJECTIVE_STYLE.get(objective, dict(color="0.35", label=str(objective), marker="o", alpha=0.45))
        ax.scatter(
            group["shot"],
            np.asarray(y_values.loc[group.index] if hasattr(y_values, "loc") else y_values[group.index]),
            s=9,
            alpha=style.get("alpha", 0.5),
            color=style["color"],
            marker=style.get("marker", "o"),
            label=style["label"],
            linewidths=0.0,
        )
    if ylabel is not None:
        ax.set_ylabel(ylabel)


readout_azimuth_values = np.mod(plot_records["readout_angle"], READOUT_AZIMUTH_PERIOD)
scatter_readout_by_objective(ax_angle, readout_azimuth_values, ylabel=r"$\varphi$")
true_phase = TRUE_DRIVE_PHASE % (2.0 * math.pi)
reference_angles = [
    (true_phase % math.pi, r"$\varphi_{\rm I}$", ":", 0.55),
    ((true_phase + 0.5 * math.pi) % math.pi, r"$\varphi_{\rm H}$", "--", 0.75),
]
for angle, label, linestyle, alpha in reference_angles:
    ax_angle.axhline(angle, color="black", lw=1.15, ls=linestyle, alpha=alpha)
    ax_angle.annotate(
        label,
        xy=(plot_records["shot"].max() * 1.01, angle),
        xytext=(0.0, 5.0),
        textcoords="offset points",
        va="bottom",
        fontsize=0.029 * paper_width_pt,
    )
draw_mixed_switch_line(ax_angle, signal_switch_shot, "#2A6FBB", label="IG→guarded Helstrom" if signal_switch_shot is not None and READOUT_OBJECTIVE == "mixed_guarded" else ("IG→Helstrom" if signal_switch_shot is not None else None))
ax_angle.set_ylim(-0.05, READOUT_AZIMUTH_PERIOD + 0.05)
if READOUT_AZIMUTH_PERIOD == math.pi:
    ax_angle.set_yticks([0, 0.5 * math.pi, math.pi])
    ax_angle.set_yticklabels([r"$0$", r"$\pi/2$", r"$\pi$"])
else:
    ax_angle.set_yticks([0, 0.5 * math.pi, math.pi, 1.5 * math.pi, 2.0 * math.pi])
    ax_angle.set_yticklabels([r"$0$", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"])
if has_polar:
    scatter_readout_by_objective(ax_polar, plot_records["readout_polar_angle"], ylabel=r"$\theta$")
    ax_polar.axhline(IDEAL_SIGNAL_POLAR_ANGLE, color="black", lw=1.0, ls="--", alpha=0.7)
    ax_polar.annotate(
        r"$\theta_{\rm O}$",
        xy=(plot_records["shot"].max() * 1.01, IDEAL_SIGNAL_POLAR_ANGLE),
        xytext=(0.0, 5.0),
        textcoords="offset points",
        va="bottom",
        fontsize=0.029 * paper_width_pt,
    )
    draw_mixed_switch_line(ax_polar, signal_switch_shot, "#2A6FBB")
    ax_polar.set_ylim(-0.05, math.pi + 0.05)
    ax_polar.set_yticks([0, 0.5 * math.pi, math.pi])
    ax_polar.set_yticklabels([r"$0$ (z)", r"$\pi/2$", r"$\pi$"])
    ax_polar.set_xlabel(r"shot label $k$", fontsize=0.035 * paper_width_pt)
else:
    ax_angle.set_xlabel(r"shot label $k$", fontsize=0.035 * paper_width_pt)
for ax in ([ax_angle, ax_polar] if ax_polar is not None else [ax_angle]):
    ax.tick_params(axis="both", which="major", labelsize=0.030 * paper_width_pt, direction="in", length=5.0, top=True, right=True)
    ax.yaxis.label.set_size(0.035 * paper_width_pt)
handles, labels = ax_angle.get_legend_handles_labels()
if handles:
    legend_ax = ax_polar if ax_polar is not None else ax_angle
    legend_ax.legend(handles, labels, loc="lower right", frameon=False, fontsize=0.027 * paper_width_pt, ncol=2)
fig.tight_layout()

pdf_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{OUTPUT_TAG}_readout_angles.pdf"
fig.savefig(pdf_path, bbox_inches="tight")
print(f"saved {pdf_path}")
display(fig)
plt.close(fig)


background_records = background_learning["records"]
background_plot_records = background_records[
    background_records["shot"] <= min(N_SHOTS, READOUT_TRAJECTORY_MAX_SHOT)
].copy()
background_switch_shot = mixed_switch_shot(background_records) if READOUT_OBJECTIVE in {"mixed", "mixed_guarded"} else None
background_has_polar = "readout_polar_angle" in background_plot_records.columns
if background_has_polar:
    fig, axes = plt.subplots(2, 1, figsize=(8.2, 5.0), sharex=True)
    ax_angle, ax_polar = axes
else:
    fig, ax_angle = plt.subplots(figsize=(8.2, 3.2))
    ax_polar = None


def scatter_background_readout_by_objective(ax, column, ylabel):
    if "active_readout_objective" not in background_plot_records.columns:
        ax.scatter(background_plot_records["shot"], background_plot_records[column], s=8, alpha=0.45, color="#536B78")
    else:
        for objective, group in background_plot_records.groupby("active_readout_objective", sort=False):
            style = OBJECTIVE_STYLE.get(objective, dict(color="0.35", label=str(objective), marker="o", alpha=0.45))
            ax.scatter(
                group["shot"],
                group[column],
                s=9,
                alpha=style.get("alpha", 0.5),
                color=style["color"],
                marker=style.get("marker", "o"),
                label=style["label"],
                linewidths=0.0,
            )
    ax.set_ylabel(ylabel)


background_plot_records["readout_azimuth_mod"] = np.mod(
    background_plot_records["readout_angle"], READOUT_AZIMUTH_PERIOD
)
scatter_background_readout_by_objective(ax_angle, "readout_azimuth_mod", r"$\varphi$")
draw_mixed_switch_line(ax_angle, background_switch_shot, "#2A6FBB", label="objective switch" if background_switch_shot is not None else None)
ax_angle.set_ylim(-0.05, READOUT_AZIMUTH_PERIOD + 0.05)
if READOUT_AZIMUTH_PERIOD == math.pi:
    ax_angle.set_yticks([0, 0.5 * math.pi, math.pi])
    ax_angle.set_yticklabels([r"$0$", r"$\pi/2$", r"$\pi$"])
else:
    ax_angle.set_yticks([0, 0.5 * math.pi, math.pi, 1.5 * math.pi, 2.0 * math.pi])
    ax_angle.set_yticklabels([r"$0$", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"])
if background_has_polar:
    scatter_background_readout_by_objective(ax_polar, "readout_polar_angle", r"$\theta$")
    draw_mixed_switch_line(ax_polar, background_switch_shot, "#2A6FBB")
    ax_polar.set_ylim(-0.05, math.pi + 0.05)
    ax_polar.set_yticks([0, 0.5 * math.pi, math.pi])
    ax_polar.set_yticklabels([r"$0$ (z)", r"$\pi/2$", r"$\pi$"])
    ax_polar.set_xlabel(r"shot label $k$", fontsize=16)
else:
    ax_angle.set_xlabel(r"shot label $k$", fontsize=16)
for ax in ([ax_angle, ax_polar] if ax_polar is not None else [ax_angle]):
    ax.tick_params(axis="both", which="major", labelsize=14, direction="in", length=5.0, top=True, right=True)
    ax.yaxis.label.set_size(16)
handles, labels = ax_angle.get_legend_handles_labels()
if handles:
    ax_angle.legend(handles, labels, loc="upper right", frameon=False, fontsize=15, ncol=2)
fig.tight_layout()

background_pdf_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{OUTPUT_TAG}_readout_angles_background.pdf"
fig.savefig(background_pdf_path, bbox_inches="tight")
print(f"saved {background_pdf_path}")
display(fig)
plt.close(fig)


In [ ]:
# Additional manuscript-ready 2x2 heatmaps.
# The shared H0 seed puts a transient finite-amplitude mode at k=256 and returns
# to a low-amplitude, nearly phase-uniform posterior by k=4096.
H0_FAKE_SIGNAL_SEED = SEED_BACKGROUND
BAYES_DRIFT_SIGNAL_SEED = int(os.environ.get("RABI_LEARNING_BAYES_DRIFT_SIGNAL_SEED", str(SEED_SIGNAL)))
H0_ADAPTIVE_STRATEGY = SOURCE_TRAJECTORY_STRATEGY
BAYES_DRIFT_STRATEGY = "adaptive_bayes_sphere_continuous"


def auxiliary_trajectory_cache_path(strategy, objective, hypothesis, seed):
    payload = setup | {
        "trajectory_kind": "paper_auxiliary_source_strategy",
        "source_trajectory_strategy": strategy,
        "readout_objective": objective,
        "hypothesis": hypothesis,
        "seed": int(seed),
    }
    objective_tag = objective.replace("/", "_")
    return CACHE_DIR / (
        f"{OUTPUT_BASENAME}_{OUTPUT_TAG}_{stable_hash(payload)}_"
        f"{objective_tag}_{hypothesis}.npz"
    )


def cached_auxiliary_trajectory(strategy, objective, hypothesis, seed):
    if (
        strategy == SOURCE_TRAJECTORY_STRATEGY
        and objective == READOUT_OBJECTIVE
    ):
        return cached_trajectory(hypothesis, seed)

    path = auxiliary_trajectory_cache_path(strategy, objective, hypothesis, seed)
    particle_grid = build_rabi_particle_grid(config.particles)
    if USE_CACHE and path.exists() and not FORCE_RECOMPUTE:
        print(f"loaded {path}")
        return load_trajectory(path, particle_grid)

    objective_tag = objective.replace("/", "_")
    legacy_path = (
        legacy_high_fidelity_cache_path(
            {
                "trajectory_kind": "paper_auxiliary_source_strategy",
                "source_trajectory_strategy": strategy,
                "readout_objective": objective,
                "hypothesis": hypothesis,
                "seed": int(seed),
            },
            f"{objective_tag}_{hypothesis}",
        )
        if USE_CACHE
        else None
    )
    if legacy_path is not None and not FORCE_RECOMPUTE:
        print(f"loaded legacy cache {legacy_path}")
        return load_trajectory(legacy_path, particle_grid)

    if os.environ.get("RABI_ALLOW_SIMULATION", "0") != "1":
        raise FileNotFoundError(f"Missing cached trajectory: {path}. Set RABI_ALLOW_SIMULATION=1 to simulate.")
    raw = run_rabi_sphere_learning_trajectory_numba(
        config,
        strategy,
        hypothesis,
        SNAPSHOT_SHOTS,
        seed=seed,
    )
    result = source_trajectory_to_result(raw, particle_grid)
    if USE_CACHE:
        save_trajectory(path, result)
        print(f"saved {path}")
    return result


background_fake_learning = cached_auxiliary_trajectory(
    H0_ADAPTIVE_STRATEGY,
    READOUT_OBJECTIVE,
    "H0",
    H0_FAKE_SIGNAL_SEED,
)
bayes_drift_signal_learning = cached_auxiliary_trajectory(
    BAYES_DRIFT_STRATEGY,
    "bayes_drift",
    "H1",
    BAYES_DRIFT_SIGNAL_SEED,
)

H0_FAKE_HEATMAP_SHOTS = tuple(
    int(x)
    for x in os.environ.get(
        "RABI_LEARNING_H0_FAKE_HEATMAP_SHOTS",
        "0,256,1024,4096",
    ).split(",")
    if x.strip()
)
BAYES_DRIFT_HEATMAP_SHOTS = tuple(
    int(x)
    for x in os.environ.get(
        "RABI_LEARNING_BAYES_DRIFT_HEATMAP_SHOTS",
        "0,256,1024,4096",
    ).split(",")
    if x.strip()
)

background_records = background_fake_learning["records"]
max_background_index = int(background_records["log_bayes_factor"].to_numpy().argmax())
print(
    "H0 fake-signal trajectory:",
    {
        "seed": H0_FAKE_SIGNAL_SEED,
        "max_log_bayes_factor": float(
            background_records.iloc[max_background_index]["log_bayes_factor"]
        ),
        "max_shot": int(background_records.iloc[max_background_index]["shot"]),
        "final_log_bayes_factor": float(background_records.iloc[-1]["log_bayes_factor"]),
    },
)
print(
    "Bayes-drift H1 final:",
    bayes_drift_signal_learning["records"]
    .iloc[-1][["h1_probability", "log_bayes_factor"]]
    .to_dict(),
)

fig_h0_fake = plot_posterior_heatmaps_paper_2x2(
    background_fake_learning,
    "heatmaps_background_fake_signal_paper_2x2",
    r"adaptive ($H_0$ data)",
    shots=H0_FAKE_HEATMAP_SHOTS,
    show_truth=False,
    color_vmax=POLICY_HEATMAP_VMAX,
)
display(fig_h0_fake)
plt.close(fig_h0_fake)

fig_bayes_drift = plot_posterior_heatmaps_paper_2x2(
    bayes_drift_signal_learning,
    "heatmaps_bayes_drift_paper_2x2",
    r"Bayes drift ($H_1$ data)",
    shots=BAYES_DRIFT_HEATMAP_SHOTS,
    show_truth=True,
)
display(fig_bayes_drift)
plt.close(fig_bayes_drift)


In [ ]:

RUN_BACKGROUND_SEED_SCAN = os.environ.get("RABI_LEARNING_RUN_BACKGROUND_SCAN", "0") == "1"
BACKGROUND_SCAN_COUNT = int(os.environ.get("RABI_LEARNING_BACKGROUND_SCAN_COUNT", "20"))

if RUN_BACKGROUND_SEED_SCAN:
    rows = []
    for offset in range(BACKGROUND_SCAN_COUNT):
        seed = SEED_BACKGROUND + offset
        result = cached_trajectory("H0", seed)
        records = result["records"]
        max_index = int(records["log_bayes_factor"].to_numpy().argmax())
        rows.append(
            {
                "seed": seed,
                "max_log_bayes_factor": float(records.iloc[max_index]["log_bayes_factor"]),
                "max_shot": int(records.iloc[max_index]["shot"]),
                "max_h1_probability": float(records.iloc[max_index]["h1_probability"]),
                "final_log_bayes_factor": float(records.iloc[-1]["log_bayes_factor"]),
                "final_h1_probability": float(records.iloc[-1]["h1_probability"]),
            }
        )
    background_scan = pd.DataFrame(rows)
    display(background_scan)
    print("fraction with max log B10 > 0:", float(np.mean(background_scan["max_log_bayes_factor"] > 0.0)))
    print("fraction with max log B10 > 1:", float(np.mean(background_scan["max_log_bayes_factor"] > 1.0)))
else:
    print("Set RABI_LEARNING_RUN_BACKGROUND_SCAN=1 to run the optional H0 seed scan.")
